<a href="https://colab.research.google.com/github/kkumarisfdc/Kiran-s_Portfolio/blob/main/Day_38_Transformers_for_DNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-

"""colab-Transformers for DNA-Dr_Bhupender-27_07_2026.ipynb

Automatically generated by Colab.

Extracting Embeddings for a gRNA using DNABERT

Section 1: Install Required Libraries

"""

# We are installing the 'transformers' library to load DNABERT,

# and 'torch' (PyTorch) which is the underlying math engine for the AI.

!pip install transformers torch

In [2]:
"""Section 2: Load the DNABERT Model and Tokenizer

Use DNABERT, we need two things:

The Tokenizer: This translates our raw DNA sequence into numbers that the model can understand.

The Model itself: This is the neural network that processes the numbers and outputs the embeddings.

We will use the 6-mer version of DNABERT. This means the model reads the DNA in "words" of 6 nucleotides at a time.

"""

# Import the necessary functions from the transformers library

from transformers import AutoTokenizer, AutoModel

import torch

# This is the exact name of the DNABERT model hosted on Hugging Face

model_name = "zhihan1996/DNA_bert_6"

# 1. Load the Tokenizer

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading tokenizer...


config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
from transformers import AutoModel

# This is the exact name of the DNABERT model hosted on Hugging Face
model_name = "zhihan1996/DNA_bert_6"

# 2. Load the Model

print("Loading model...")

model = AutoModel.from_pretrained(model_name)

print("DNABERT is successfully loaded and ready to go!")

Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_6
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DNABERT is successfully loaded and ready to go!


In [5]:
"""Section 3: Prepare the gRNA Sequence (K-mer Tokenization)

A standard gRNA (guide RNA) used in CRISPR is usually 20 nucleotides long.

DNABERT cannot read a continuous string like "ATCGATCG...". It requires the sequence to be broken down into overlapping overlapping chunks called k-mers. Since we are using DNABERT-6, we need to create 6-mers.

For example, ATCGAA becomes ATCGAA TCGAAA CGAAAT...

"""

# 1. Define the sequence

grna_sequence = "ATCGATCGCGCATAGCGCAT"

# 2. Create an empty list

kmers = []

# 3. Loop through the sequence and extract 6-mers directly

for i in range(len(grna_sequence) - 5):

  kmers.append(grna_sequence[i:i+6])

# 4. Join them with spaces

kmer_sequence = " ".join(kmers)

print(kmer_sequence)

ATCGAT TCGATC CGATCG GATCGC ATCGCG TCGCGC CGCGCA GCGCAT CGCATA GCATAG CATAGC ATAGCG TAGCGC AGCGCA GCGCAT


In [10]:
from transformers import AutoTokenizer

"""Section 4: Convert K-mers to Model Inputs (Tokenization)

Now that we have our spaced 6-mers, we need to use the Tokenizer we loaded earlier. The tokenizer will convert these textual "words" into specific ID numbers. It will also add special hidden tokens like [CLS] (start of sequence) and [SEP] (end of sequence) that the model requires.

"""

# Ensure model_name is defined for the tokenizer
model_name = "zhihan1996/DNA_bert_6"

# 1. Load the Tokenizer (if not already loaded)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Pass our k-mer sequence into the tokenizer

# return_tensors="pt" tells it to return PyTorch format (which the model needs)

inputs = tokenizer(kmer_sequence, return_tensors="pt")

# Let's see what the tokenizer did to our sequence

print("Token IDs (Numbers the model will read):")

print(inputs["input_ids"])

# A quick check to see how many tokens we have

num_tokens = inputs["input_ids"].shape[1]

print(f"\nTotal number of tokens generated: {num_tokens}")

Token IDs (Numbers the model will read):
tensor([[   2,  438, 1739, 2848, 3187,  448, 1779, 3005, 3814, 2953, 3608, 2131,
          320, 1267,  957, 3814,    3]])

Total number of tokens generated: 17


In [11]:
import torch

"""Section 5: Extract the Embeddings!

We will now feed our tokenized inputs into the DNABERT model.

We use torch.no_grad() to tell the computer we are just using the model, not training it. This saves a lot of memory and time.

The model will output a complex object, but what we want is the last_hidden_state. This is the final mathematical representation (embedding) of our gRNA after it has passed through all of DNABERT's attention layers.

"""

# Turn off gradient calculation to save memory (we aren't training the model)

with torch.no_grad():

# Pass the inputs into the model

  outputs = model(**inputs)

# Extract the embeddings from the last layer of the model

embeddings = outputs.last_hidden_state

print("Embeddings successfully extracted!")

Embeddings successfully extracted!


In [12]:


"""Section 6: Understanding the Output Shape

What exactly did we just generate? Let's look at the shape (dimensions) of our embedding matrix.

It will output 3 numbers: [Batch Size, Sequence Length, Hidden Dimension].

Batch Size: How many sequences we processed at once (1).

Sequence Length: How many tokens were in our sequence (including [CLS] and [SEP]).

Hidden Dimension: The size of the mathematical vector used to describe each token (For DNABERT-6, this is usually 768).

"""

# Check the shape of our final embeddings tensor

print("Shape of the embeddings tensor:", embeddings.shape)

# Let's extract just the vector representing the very first token ([CLS])

# The [CLS] token is special—it is often used as a summary of the ENTIRE sequence!

cls_embedding = embeddings[0, 0, : ]

print("\nShape of the [CLS] sequence summary vector:", cls_embedding.shape)

print("\nHere are the first 10 numbers of our gRNA's mathematical embedding:")

print(cls_embedding[:10])

Shape of the embeddings tensor: torch.Size([1, 17, 768])

Shape of the [CLS] sequence summary vector: torch.Size([768])

Here are the first 10 numbers of our gRNA's mathematical embedding:
tensor([-1.5445, -0.9196, -0.6993, -0.8491, -0.0123, -0.9346, -1.0870, -0.4152,
         1.6766, -1.0143])
